# Step 1 : Thu thập dữ liệu
Đọc bộ Customer Shopping Trends từ Kaggle.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
df=pd.read_csv(Path('data/raw/shopping_trends.csv'))
df.head()

# Step 2 : Thống kê và trực quan hóa dữ liệu

In [ ]:
df.info()
df.describe(include='all').T

In [ ]:
fig,axes=plt.subplots(1,5,figsize=(20,4))
for ax,col in zip(axes,['Purchase Amount (USD)','Age','Review Rating','Previous Purchases','Category']):
    if df[col].dtype == 'object':
        df.groupby(col)['Purchase Amount (USD)'].mean().plot.bar(ax=ax)
    else:
        pd.to_numeric(df[col],errors='coerce').plot.hist(ax=ax,bins=15)
    ax.set_title(col)
plt.tight_layout()

# Kết luận
Dữ liệu không có mốc thời gian; mỗi dòng là một giao dịch.

# Step 3 : Tiền xử lí dữ liệu

## Fill các giá trị Missing

In [ ]:
df=df.copy()
for c in df.select_dtypes(include='object'): df[c]=df[c].fillna('__MISSING__')
for c in df.select_dtypes(exclude='object'): df[c]=df[c].fillna(df[c].median())

## Feature engineering

In [ ]:
frequency_map={'Weekly':52,'Fortnightly':26,'Bi-Weekly':26,'Monthly':12,'Every 3 Months':4,'Quarterly':4,'Annually':1}
df['annual_purchase_frequency']=df['Frequency of Purchases'].map(frequency_map).fillna(0)
df['product_sales_amount_usd']=df['Purchase Amount (USD)']

## Tiền xử lí dữ liệu
Tách ngẫu nhiên 70/15/15 với seed 42 và fit encoder/scaler trên train.

# Step 4 : Train model

In [ ]:
from src.pipeline import prepare_data
prepared=prepare_data('data/raw')
prepared['X_train'].shape

# Step 5 : Đánh giá

In [ ]:
from src.metrics import regression_metrics
print('Tính MAE, MSE, RMSE và R² trên tập kiểm tra.')